# Phase 06 — JobFitAlignment Training Experiments

**Status:** Complete  
**Workflow:** Notebook-first training documentation  
**Purpose:** Plan and compare model experiments for jobFitAlignment scoring and ranking signals.

This notebook is the Phase 6 source of truth. It writes `reports/phase_06_jobfit_training_experiments.json` with the experiment matrix, input feature contract, output signal contract, training protocol, selection gates, and current no-train blocker decision.


## Contract boundary

Phase 6 owns the jobFitAlignment experiment plan and model-core signal contract only. It does not train a production model while Phase 5 readiness is a no-go, does not create backend response copy, does not produce `topActionables`, does not produce `sectionReviews`, and does not hydrate job details.

The model core may emit grounded job-fit signals. The API wrapper remains responsible for converting those signals into the public `jobFitAlignment.summary` string and for composing wrapper-owned product copy.


## Shared setup

### Purpose
Load prior phase reports and the generated OpenAPI contract, then define helpers used by every Phase 6 step.

### Required input
Repository root with `GAP_MODEL_TRAINING.md`, `reports/phase_02_label_schema_baselines.json`, `reports/phase_03_normalization_feature_design.json`, `reports/phase_04_pair_generation_splits.json`, `reports/phase_05_baseline_evaluation.json`, and `references/docs/generated/openapi.json`.

### Action
Read prior decisions, capture current readiness blockers, extract the public jobFitAlignment response shape, and keep all Phase 6 outputs machine-readable.

### Expected output
Reusable variables for baseline evidence, blocker inheritance, OpenAPI score constraints, and final report writing.

### Verification
Fail fast if any required prior report is missing. Confirm the OpenAPI jobFitAlignment public score remains constrained to `0-100`.


In [13]:
from __future__ import annotations

import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "GAP_MODEL_TRAINING.md").exists() and (candidate / "training").exists():
            return candidate
    raise RuntimeError("Could not locate repository root from notebook runtime.")


ROOT = find_repo_root(Path.cwd())
REPORTS = ROOT / "reports"
OPENAPI_PATH = ROOT / "references" / "docs" / "generated" / "openapi.json"


def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Required Phase 6 input is missing: {path.relative_to(ROOT)}")
    return json.loads(path.read_text())


phase2 = read_json(REPORTS / "phase_02_label_schema_baselines.json")
phase3 = read_json(REPORTS / "phase_03_normalization_feature_design.json")
phase4 = read_json(REPORTS / "phase_04_pair_generation_splits.json")
phase5 = read_json(REPORTS / "phase_05_baseline_evaluation.json")
openapi = read_json(OPENAPI_PATH)

public_jobfit_schema = (
    openapi["components"]["schemas"]["CvAnalysis"]["properties"]["analysisResult"]["properties"]["jobFitAlignment"]
)
public_score_schema = public_jobfit_schema["properties"]["score"]
assert public_score_schema["minimum"] == 0
assert public_score_schema["maximum"] == 100

phase5_blockers = list(phase5.get("blocked_until_later_phases", []))
phase5_gate = phase5.get("training_readiness_gate", {})
if phase5_gate.get("blockers"):
    phase5_blockers.extend(phase5_gate["blockers"])

regression_baseline_section = phase5.get("regression_baselines", {})
if isinstance(regression_baseline_section, dict):
    regression_baseline_rows = regression_baseline_section.get("metrics", [])
else:
    regression_baseline_rows = regression_baseline_section

validation_baselines = [
    row for row in regression_baseline_rows
    if isinstance(row, dict) and row.get("split") == "validation"
]
best_baseline = phase5_gate.get("best_validation_metrics") or (
    sorted(
        validation_baselines,
        key=lambda row: (row.get("mae", float("inf")), row.get("rmse", float("inf"))),
    )[0] if validation_baselines else None
)

setup_summary = {
    "public_jobfit_schema": public_jobfit_schema,
    "best_validation_baseline": best_baseline,
    "phase5_readiness_decision": phase5_gate.get("decision", "unknown"),
    "inherited_blockers": sorted(set(phase5_blockers)),
}
setup_summary


{'public_jobfit_schema': {'type': 'object',
  'additionalProperties': False,
  'required': ['score', 'summary'],
  'properties': {'score': {'type': 'integer', 'minimum': 0, 'maximum': 100},
   'summary': {'type': 'string'}}},
 'best_validation_baseline': {'baseline': 'simple_ridge_regression',
  'mae': 0.01726078006981349,
  'r2': 0.9579090970833964,
  'rmse': 0.021016879423409764,
  'score_band_agreement': 0.992434988179669,
  'spearman': 0.9649904957896672,
  'split': 'validation'},
 'phase5_readiness_decision': 'NO_GO_FIX_PAIR_GENERATION_AND_LABELS_FIRST',
 'inherited_blockers': ['Balanced Phase 4 pair metadata columns are missing from the available pair artifact.',
  'Human-labeled validation data is not available; legacy fit_score remains weak-label prototype evidence.',
  'Phase 10 calibration must wait for stable label distribution and model outputs.',
  'Phase 6 complex job-fit training must wait for balanced pairs, high-fit coverage, and stronger labels.',
  'Phase 9 recommend

## Step 6.1 — Experiment matrix

### Purpose
Define models to compare: linear baseline, cosine-only baseline, neural scorer, feature-augmented scorer, and optional ranking objective.

### Required input
Use Phase 5 baseline evidence, Phase 4 split policy, Phase 3 normalized feature rules, and Phase 2 label/band policy. Balanced pairs with high-fit examples are required before any complex model is trained.

### Action
Create a fixed experiment catalog with stable IDs, objectives, required inputs, comparable metrics, training status, and blocker policy. Baselines remain the reference floor; neural and ranking experiments stay planned until the readiness gate passes.

### Expected output
A complete experiment matrix that reviewers can compare without reading code internals.

### Verification
Every required model family is present exactly once or more, each experiment has comparable metrics, and no experiment is marked trainable while inherited blockers are unresolved.


In [14]:
common_regression_metrics = ["mae", "rmse", "r2", "spearman", "score_band_agreement"]
common_ranking_metrics = ["ndcg_at_5", "ndcg_at_10", "map_at_10"]

experiment_matrix = [
    {
        "experiment_id": "jobfit_linear_feature_baseline_v1",
        "model_family": "linear_baseline",
        "objective": "Regression on transparent normalized scalar features.",
        "required_features": ["skill_overlap_score", "semantic_similarity_score", "experience_match_score", "role_match_score", "requirement_coverage_score"],
        "label_source": "Phase 2 job-fit score; weak labels allowed only as prototype evidence.",
        "metrics": common_regression_metrics,
        "comparison_role": "reference_floor",
        "train_status": "reference_only_until_balanced_pairs_exist",
        "promotion_gate": "Must remain in report as a minimum floor for all complex scorers.",
    },
    {
        "experiment_id": "jobfit_cosine_only_baseline_v1",
        "model_family": "cosine_only_baseline",
        "objective": "Regression/ranking using only profile-CV and job text embedding cosine similarity.",
        "required_features": ["profile_cv_embedding", "job_embedding", "embedding_model_version"],
        "label_source": "Same split and labels as complex scorers.",
        "metrics": common_regression_metrics + common_ranking_metrics,
        "comparison_role": "semantic_similarity_floor",
        "train_status": "planned_no_complex_training",
        "promotion_gate": "Complex models must beat this baseline on regression and ranking metrics.",
    },
    {
        "experiment_id": "jobfit_neural_embedding_scorer_v1",
        "model_family": "neural_scorer",
        "objective": "Learn non-linear interactions between profile-CV embedding and job embedding for score prediction.",
        "required_features": ["profile_cv_embedding", "job_embedding", "embedding_difference", "embedding_product"],
        "label_source": "Balanced Phase 4/next pair artifact with low/medium/high coverage and group-safe splits.",
        "metrics": common_regression_metrics + common_ranking_metrics,
        "comparison_role": "candidate_complex_scorer",
        "train_status": "blocked_by_phase5_no_go",
        "promotion_gate": "Eligible only after it beats best baseline and passes slice stability gates.",
    },
    {
        "experiment_id": "jobfit_feature_augmented_scorer_v1",
        "model_family": "feature_augmented_scorer",
        "objective": "Combine embedding interactions with normalized skill, experience, role, language, and requirement coverage features.",
        "required_features": ["profile_cv_embedding", "job_embedding", "skill_overlap_score", "experience_gap", "role_match_score", "requirement_coverage_score", "language", "pair_type"],
        "label_source": "Balanced pair labels plus manual validation labels when available.",
        "metrics": common_regression_metrics + common_ranking_metrics + ["slice_mae_delta", "high_fit_recall"],
        "comparison_role": "primary_jobfit_candidate",
        "train_status": "blocked_by_phase5_no_go",
        "promotion_gate": "Primary candidate if it clears global, ranking, and slice gates without unsupported signals.",
    },
    {
        "experiment_id": "jobfit_pairwise_ranking_objective_v1",
        "model_family": "ranking_objective",
        "objective": "Optimize ordering among backend-provided candidate jobs for each profile/CV context.",
        "required_features": ["candidate_set_id", "profile_id", "job_id", "profile_cv_embedding", "job_embedding", "candidate_relevance_label", "candidate_rank_group"],
        "label_source": "Backend-like candidate sets with relevance labels or manual ranking review.",
        "metrics": common_ranking_metrics + ["unknown_job_id_rate", "duplicate_job_id_rate"],
        "comparison_role": "optional_ranking_experiment",
        "train_status": "blocked_until_candidate_sets_and_relevance_labels_exist",
        "promotion_gate": "Cannot replace score model unless candidate-set constraints and ranking uplift pass.",
    },
]

required_families = {"linear_baseline", "cosine_only_baseline", "neural_scorer", "feature_augmented_scorer", "ranking_objective"}
observed_families = {row["model_family"] for row in experiment_matrix}
assert required_families <= observed_families
assert all(row["metrics"] for row in experiment_matrix)
assert all("blocked" in row["train_status"] or "reference" in row["train_status"] or "planned" in row["train_status"] for row in experiment_matrix)
experiment_matrix


[{'experiment_id': 'jobfit_linear_feature_baseline_v1',
  'model_family': 'linear_baseline',
  'objective': 'Regression on transparent normalized scalar features.',
  'required_features': ['skill_overlap_score',
   'semantic_similarity_score',
   'experience_match_score',
   'role_match_score',
   'requirement_coverage_score'],
  'label_source': 'Phase 2 job-fit score; weak labels allowed only as prototype evidence.',
  'metrics': ['mae', 'rmse', 'r2', 'spearman', 'score_band_agreement'],
  'comparison_role': 'reference_floor',
  'train_status': 'reference_only_until_balanced_pairs_exist',
  'promotion_gate': 'Must remain in report as a minimum floor for all complex scorers.'},
 {'experiment_id': 'jobfit_cosine_only_baseline_v1',
  'model_family': 'cosine_only_baseline',
  'objective': 'Regression/ranking using only profile-CV and job text embedding cosine similarity.',
  'required_features': ['profile_cv_embedding',
   'job_embedding',
   'embedding_model_version'],
  'label_source': 

## Step 6.2 — Input feature contract

### Purpose
Document required inputs: profile/CV embedding, job embedding, normalized skill overlap, experience gap, role match, and requirement coverage.

### Required input
Use Phase 3 normalization rules, Phase 4 pair metadata schema, and Phase 5 baseline feature evidence. Feature values must be derived from source profile/CV/job content, not public response copy or backend-owned hydrated details.

### Action
Define each required feature name, type, range or shape, source evidence, missing-value policy, leakage rule, and validation check.

### Expected output
A machine-readable input feature contract for future training code and model-card export.

### Verification
All required feature names from the phase plan are covered. Unknown values are preserved as explicit `UNKNOWN`/null indicators instead of silently folded into common classes.


In [15]:
input_feature_contract = [
    {
        "feature": "profile_cv_embedding",
        "type": "float_vector",
        "range_or_shape": "embedding_dim from embedding_model_version; L2-normalized before cosine features",
        "source_evidence": "profile text plus parsed CV text when available",
        "missing_policy": "fail training row if both profile and CV text are empty; otherwise record source_text_coverage",
        "leakage_rule": "must not include target score, public API summary, manual validation note, or backend product copy",
        "validation_check": "embedding exists, finite values only, embedding_model_version recorded",
    },
    {
        "feature": "job_embedding",
        "type": "float_vector",
        "range_or_shape": "same dimension and model version as profile_cv_embedding",
        "source_evidence": "job title, role, required skills, requirements, and description text",
        "missing_policy": "fail row if job text is empty; keep stale/inactive job filtering outside model training unless source snapshot labels it",
        "leakage_rule": "must not include backend hydration fields unavailable to model core at inference",
        "validation_check": "same embedding_model_version as profile_cv_embedding",
    },
    {
        "feature": "normalized_skill_overlap",
        "type": "float",
        "range_or_shape": "0.0-1.0",
        "source_evidence": "Phase 3 normalized profile/CV skills and job skills",
        "missing_policy": "empty skill lists produce 0.0 plus empty_skills flags",
        "leakage_rule": "skill aliases must come from versioned dictionary, not validation labels",
        "validation_check": "all raw skills normalized with alias version and unknown-skill count reported",
    },
    {
        "feature": "experience_gap",
        "type": "float",
        "range_or_shape": "candidate years minus required years; clipped diagnostic buckets allowed for model input",
        "source_evidence": "Phase 3 mapped experience values",
        "missing_policy": "UNKNOWN experience gets explicit missing indicator and conservative gap feature",
        "leakage_rule": "must not infer seniority from target label or output score",
        "validation_check": "no observed Phase 3 experience value falls through silently",
    },
    {
        "feature": "role_match",
        "type": "float_or_category",
        "range_or_shape": "0.0-1.0 score plus optional role_family category",
        "source_evidence": "normalized profile target role, CV role evidence, job title, and job role family",
        "missing_policy": "missing profile role becomes UNKNOWN role family and score 0.0 unless CV role evidence exists",
        "leakage_rule": "must not use job recommendation rank or backend search position",
        "validation_check": "role family vocabulary version recorded",
    },
    {
        "feature": "requirement_coverage",
        "type": "float",
        "range_or_shape": "0.0-1.0",
        "source_evidence": "job requirements matched against profile/CV skills, tools, education, experience, and evidence text",
        "missing_policy": "empty job requirements produce null coverage and missing_requirement flag, not 1.0",
        "leakage_rule": "manual reviewer comments must never become coverage features",
        "validation_check": "coverage denominator and unmatched requirement count recorded",
    },
    {
        "feature": "pair_metadata",
        "type": "object",
        "range_or_shape": "pair_type, split, profile_id, job_id, language, score_band, label_version",
        "source_evidence": "Phase 4 pair-generation and split policy",
        "missing_policy": "training blocked if split, pair_type, label_version, or score_band is missing",
        "leakage_rule": "IDs may be used for grouping/audit only, never as predictive model features",
        "validation_check": "profile_id is isolated across splits and high-fit coverage exists in validation/test",
    },
]

required_feature_terms = {"profile_cv_embedding", "job_embedding", "normalized_skill_overlap", "experience_gap", "role_match", "requirement_coverage"}
observed_feature_terms = {row["feature"] for row in input_feature_contract}
assert required_feature_terms <= observed_feature_terms
assert all(row["missing_policy"] and row["leakage_rule"] and row["validation_check"] for row in input_feature_contract)
input_feature_contract


[{'feature': 'profile_cv_embedding',
  'type': 'float_vector',
  'range_or_shape': 'embedding_dim from embedding_model_version; L2-normalized before cosine features',
  'source_evidence': 'profile text plus parsed CV text when available',
  'missing_policy': 'fail training row if both profile and CV text are empty; otherwise record source_text_coverage',
  'leakage_rule': 'must not include target score, public API summary, manual validation note, or backend product copy',
  'validation_check': 'embedding exists, finite values only, embedding_model_version recorded'},
 {'feature': 'job_embedding',
  'type': 'float_vector',
  'range_or_shape': 'same dimension and model version as profile_cv_embedding',
  'source_evidence': 'job title, role, required skills, requirements, and description text',
  'missing_policy': 'fail row if job text is empty; keep stale/inactive job filtering outside model training unless source snapshot labels it',
  'leakage_rule': 'must not include backend hydration

## Step 6.3 — Output signal contract

### Purpose
Define model-owned outputs: score, summarySignals, missingSignals, matchedSkills, missingSkills, and confidence notes.

### Required input
Use the Phase 1 model/API boundary, OpenAPI public `jobFitAlignment` score/summary shape, Phase 2 score band policy, and Phase 3 normalized skill evidence.

### Action
Define a model-core signal object that can be transformed into the public API summary by the wrapper without inventing unsupported claims.

### Expected output
A durable output signal contract with field ownership, type, score range, grounding rules, and wrapper mapping.

### Verification
The model-core contract must stay inside the model boundary. `topActionables`, `sectionReviews`, job detail hydration, auth, persistence, and final product copy stay outside this notebook.


In [16]:
output_signal_contract = {
    "schema_version": "jobfit-alignment-core-v1",
    "public_api_mapping": {
        "jobFitAlignment.score": "core.score rounded/clipped to integer 0-100 after Phase 10 calibration",
        "jobFitAlignment.summary": "API wrapper renders from summarySignals, missingSignals, confidenceNotes, matchedSkills, and missingSkills",
    },
    "model_owned_fields": [
        {
            "field": "score",
            "type": "integer",
            "range": [0, 100],
            "grounding_rule": "Derived from calibrated job-fit model output only; no wrapper sentiment or backend ranking position.",
        },
        {
            "field": "summarySignals",
            "type": "array[string]",
            "range": "0-8 short evidence keys",
            "grounding_rule": "Only observed alignment evidence such as role_match, strong_skill_overlap, experience_match, and requirement_coverage.",
        },
        {
            "field": "missingSignals",
            "type": "array[string]",
            "range": "0-8 short evidence keys",
            "grounding_rule": "Only observed gaps such as missing_required_skill, experience_gap, low_requirement_coverage, or unknown_language.",
        },
        {
            "field": "matchedSkills",
            "type": "array[string]",
            "range": "0-20 normalized display skills",
            "grounding_rule": "Intersection of normalized candidate skills and job skills/requirements with source evidence.",
        },
        {
            "field": "missingSkills",
            "type": "array[string]",
            "range": "0-20 normalized display skills",
            "grounding_rule": "Required job skills not found in normalized profile/CV evidence.",
        },
        {
            "field": "confidenceNotes",
            "type": "array[string]",
            "range": "0-5 diagnostic notes",
            "grounding_rule": "Data-quality and model-confidence notes such as low_text_coverage, unknown_experience, unknown_language, or out_of_distribution_role.",
        },
    ],
    "not_model_owned_fields": ["topActionables", "sectionReviews", "overallImpression", "jobRecommendations.title", "jobRecommendations.companyName", "auth", "persistence", "hydratedJobDetails"],
}

owned_field_names = {field["field"] for field in output_signal_contract["model_owned_fields"]}
required_output_fields = {"score", "summarySignals", "missingSignals", "matchedSkills", "missingSkills", "confidenceNotes"}
assert required_output_fields == owned_field_names
assert output_signal_contract["model_owned_fields"][0]["range"] == [public_score_schema["minimum"], public_score_schema["maximum"]]
assert "topActionables" in output_signal_contract["not_model_owned_fields"]
output_signal_contract


{'schema_version': 'jobfit-alignment-core-v1',
 'public_api_mapping': {'jobFitAlignment.score': 'core.score rounded/clipped to integer 0-100 after Phase 10 calibration',
  'jobFitAlignment.summary': 'API wrapper renders from summarySignals, missingSignals, confidenceNotes, matchedSkills, and missingSkills'},
 'model_owned_fields': [{'field': 'score',
   'type': 'integer',
   'range': [0, 100],
   'grounding_rule': 'Derived from calibrated job-fit model output only; no wrapper sentiment or backend ranking position.'},
  {'field': 'summarySignals',
   'type': 'array[string]',
   'range': '0-8 short evidence keys',
   'grounding_rule': 'Only observed alignment evidence such as role_match, strong_skill_overlap, experience_match, and requirement_coverage.'},
  {'field': 'missingSignals',
   'type': 'array[string]',
   'range': '0-8 short evidence keys',
   'grounding_rule': 'Only observed gaps such as missing_required_skill, experience_gap, low_requirement_coverage, or unknown_language.'},


## Step 6.4 — Training protocol

### Purpose
Document split use, seeds, batch strategy, early stopping policy, checkpoint naming, and failure recovery expectations.

### Required input
Use Phase 4 group isolation rules, Phase 5 no-go gate, and the experiment matrix. A future runnable training job must use only approved inputs and must produce model artifacts with report evidence.

### Action
Define the canonical run protocol before implementation: split contract, random seeds, batch/evaluation cadence, early stopping, artifact naming, recovery behavior, and blocker handling.

### Expected output
A training protocol that future code can implement without changing acceptance criteria.

### Verification
Protocol must block complex training while Phase 5 blockers exist and must require deterministic run metadata for every artifact.


In [17]:
training_protocol = {
    "current_decision": "do_not_train_complex_models",
    "decision_reason": "Phase 5 readiness gate is no-go; balanced pairs, high-fit validation/test examples, and human validation labels are missing.",
    "split_use": {
        "train": "fit model parameters only",
        "validation": "select hyperparameters, early stopping, and best checkpoint",
        "test": "single final evaluation after model selection",
        "group_isolation": phase4.get("split_policy", {}).get("group_isolation", "profile_id must not cross splits"),
    },
    "seeds": [20260601, 20260617, 20260701],
    "batch_strategy": {
        "default_batch_size": 256,
        "stratification": "preserve score_band and pair_type coverage when feasible",
        "ranking_batches": "group candidate jobs by profile_id/candidate_set_id for pairwise or listwise objectives",
    },
    "early_stopping_policy": {
        "monitor": "validation_mae for score models; validation_ndcg_at_10 for ranking models",
        "patience_epochs": 5,
        "min_delta": 0.001,
        "restore_best_weights": True,
        "max_epochs_without_review": 50,
    },
    "checkpoint_naming": {
        "model_path_template": "models/jobfit_alignment/{experiment_id}/{run_id}/model.keras",
        "report_path_template": "reports/jobfit_alignment/{experiment_id}/{run_id}/metrics.json",
        "run_id_fields": ["date", "label_version", "feature_config_version", "split_seed", "git_commit"],
    },
    "failure_recovery": [
        "Fail before training when required feature columns, split names, label_version, or high-fit validation/test rows are missing.",
        "Keep failed run reports with error reason and input hashes; do not overwrite best checkpoints.",
        "Resume only from immutable dataset hash and matching feature config version.",
        "Never mark gate passed if only weak-label prototype metrics are available.",
    ],
    "training_blockers": sorted(set(phase5_blockers)),
}

assert training_protocol["current_decision"] == "do_not_train_complex_models"
assert training_protocol["seeds"]
assert "model_path_template" in training_protocol["checkpoint_naming"]
training_protocol


{'current_decision': 'do_not_train_complex_models',
 'decision_reason': 'Phase 5 readiness gate is no-go; balanced pairs, high-fit validation/test examples, and human validation labels are missing.',
 'split_use': {'train': 'fit model parameters only',
  'validation': 'select hyperparameters, early stopping, and best checkpoint',
  'test': 'single final evaluation after model selection',
  'group_isolation': 'profile_id must not cross splits'},
 'seeds': [20260601, 20260617, 20260701],
 'batch_strategy': {'default_batch_size': 256,
  'stratification': 'preserve score_band and pair_type coverage when feasible',
  'ranking_batches': 'group candidate jobs by profile_id/candidate_set_id for pairwise or listwise objectives'},
 'early_stopping_policy': {'monitor': 'validation_mae for score models; validation_ndcg_at_10 for ranking models',
  'patience_epochs': 5,
  'min_delta': 0.001,
  'restore_best_weights': True,
  'max_epochs_without_review': 50},
 'checkpoint_naming': {'model_path_templ

## Step 6.5 — Selection criteria

### Purpose
Define minimum improvement over the best baseline, including MAE improvement, R-squared, Spearman, and slice stability.

### Required input
Use Phase 5 best-baseline metrics, GAP production gates, Phase 4 diagnostic gates, and Phase 2 score-band policy.

### Action
Create explicit model-selection gates for global metrics, ranking quality, slice stability, output contract safety, and artifact readiness.

### Expected output
A go/no-go gate definition and machine-readable Phase 6 report.

### Verification
Selection criteria must be stricter than legacy prototype gates and must prevent promotion when high-fit coverage, human validation, calibration, or contract safety is missing.


In [18]:
best_baseline_mae = float(best_baseline["mae"]) if best_baseline and best_baseline.get("mae") is not None else None
best_baseline_name = best_baseline["baseline"] if best_baseline else None

selection_criteria = {
    "baseline_reference": {
        "best_validation_baseline": best_baseline_name,
        "best_validation_mae": best_baseline_mae,
        "source_report": "reports/phase_05_baseline_evaluation.json",
    },
    "minimum_global_gates": {
        "mae_improvement_vs_best_baseline": ">= 20%",
        "r2": ">= 0.15 on validation and positive on test",
        "spearman": ">= 0.35 on validation and test",
        "score_band_agreement": ">= best baseline and reported by low/medium/high band",
    },
    "ranking_gates": {
        "ndcg_at_10": ">= best ranking baseline + 15% when candidate labels exist",
        "map_at_10": ">= best ranking baseline and reported by candidate-set size",
        "candidate_constraints": "unknown_job_id_rate == 0 and duplicate_job_id_rate == 0 for backend-provided candidate sets",
    },
    "slice_stability_gates": {
        "required_slices": ["role_family", "language", "experience_band", "pair_type", "score_band"],
        "max_slice_regression": "no critical slice may regress by more than 10% MAE versus best baseline without documented exception",
        "minimum_high_fit_coverage": phase4.get("minimum_high_fit_coverage"),
        "language_policy": "ID, EN, MIXED, and UNKNOWN reported separately; UNKNOWN is never hidden inside EN",
    },
    "contract_safety_gates": {
        "score_range": "all scores clipped/calibrated to integer 0-100",
        "grounding": "matchedSkills and missingSkills must appear in normalized source evidence",
        "boundary": "model output must not include topActionables, sectionReviews, hydrated job details, auth, or persistence fields",
        "calibration_dependency": "Phase 10 calibration required before production score semantics are claimed",
    },
    "current_gate_result": {
        "decision": "NO-GO for training run; GO for experiment plan completion",
        "reasons": sorted(set(phase5_blockers)),
    },
}

acceptance = {
    "experiment_matrix_complete_before_code_is_run": True,
    "output_signal_contract_matches_product_boundary": True,
    "model_selection_gates_are_explicit": True,
}

phase6_report = {
    "schema_version": "phase-06-jobfit-training-experiments-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "inputs": {
        "phase2_report": "reports/phase_02_label_schema_baselines.json",
        "phase3_report": "reports/phase_03_normalization_feature_design.json",
        "phase4_report": "reports/phase_04_pair_generation_splits.json",
        "phase5_report": "reports/phase_05_baseline_evaluation.json",
        "openapi_json": "references/docs/generated/openapi.json",
    },
    "setup_summary": setup_summary,
    "experiment_matrix": experiment_matrix,
    "input_feature_contract": input_feature_contract,
    "output_signal_contract": output_signal_contract,
    "training_protocol": training_protocol,
    "selection_criteria": selection_criteria,
    "blocked_until_later_phases": [
        "Complex JobFitAlignment training must wait for a materialized balanced pair dataset with high-fit validation/test coverage.",
        "Promotion must wait for human validation labels or equivalent trusted validation evidence.",
        "Production score semantics must wait for Phase 10 calibration.",
        "Ranking-objective promotion must wait for backend-like candidate sets and relevance labels.",
    ],
    "acceptance": acceptance,
}

report_path = REPORTS / "phase_06_jobfit_training_experiments.json"
report_path.write_text(json.dumps(phase6_report, indent=2, sort_keys=True) + "\n")
phase6_report


{'schema_version': 'phase-06-jobfit-training-experiments-v1',
 'generated_at_utc': '2026-06-01T09:22:10.537866+00:00',
 'inputs': {'phase2_report': 'reports/phase_02_label_schema_baselines.json',
  'phase3_report': 'reports/phase_03_normalization_feature_design.json',
  'phase4_report': 'reports/phase_04_pair_generation_splits.json',
  'phase5_report': 'reports/phase_05_baseline_evaluation.json',
  'openapi_json': 'references/docs/generated/openapi.json'},
 'setup_summary': {'public_jobfit_schema': {'type': 'object',
   'additionalProperties': False,
   'required': ['score', 'summary'],
   'properties': {'score': {'type': 'integer', 'minimum': 0, 'maximum': 100},
    'summary': {'type': 'string'}}},
  'best_validation_baseline': {'baseline': 'simple_ridge_regression',
   'mae': 0.01726078006981349,
   'r2': 0.9579090970833964,
   'rmse': 0.021016879423409764,
   'score_band_agreement': 0.992434988179669,
   'spearman': 0.9649904957896672,
   'split': 'validation'},
  'phase5_readiness_

## Acceptance criteria

- [x] Experiment matrix is complete before code is run.
- [x] Output signal contract matches product boundary.
- [x] Model selection gates are explicit.


## Phase notes

- Phase 6 completes the jobFitAlignment experiment plan and signal contracts.
- Complex training is intentionally not run because Phase 5 readiness is **NO-GO**.
- Future training must first materialize balanced pairs, high-fit validation/test coverage, trusted validation labels, and backend-like candidate ranking evidence.
